# Layer 1: Reseller demand
Estimate random-coefficients BLP on requested orders. Markets are category-region-quarter-ordering period. The outside option is explicit. See README_LAYER1.md before interpreting recovery.

In [ ]:
from pathlib import Path
import pandas as pd
from layer1 import Layer1

DATA = Path("data")
OUTPUT = Path("results/layer1")
OUTPUT.mkdir(parents=True, exist_ok=True)
model = Layer1(DATA)
display(model.products.head())
print("Markets:", model.M, "Products per market:", model.J)

## Estimate
The linear utility coefficients are concentrated out by IV. Optimize quality random-coefficient standard deviation and its size interaction. Two-step weighting clusters monthly moment sums within category-region-quarter. This may take several minutes.

In [ ]:
coefficients, product_results, diagnostics, first_stage = model.fit()
display(coefficients)
display(first_stage)
print(diagnostics)

## Predicted reseller demand
R_hat = capacity × predicted share. Unobserved reseller tastes are integrated out conditional on size. These are conditional-mean predictions, not recovered individual latent tastes.

In [ ]:
predictions = model.predict()
display(predictions.head())
layer2_panel = pd.read_csv(DATA / "reseller_panel.csv").merge(
    predictions[["reseller_id", "sku", "market_period_id", "R_hat"]],
    on=["reseller_id", "sku", "market_period_id"], how="left", validate="one_to_one"
)
assert layer2_panel.R_hat.notna().all()
display(layer2_panel.head())

## Save
These outputs feed Layer 2. No true-demand columns enter estimation.

In [ ]:
import json
coefficients.to_csv(OUTPUT / "coefficients.csv", index=False)
product_results.to_csv(OUTPUT / "product_market_results.csv", index=False)
predictions.to_csv(OUTPUT / "reseller_demand_predictions.csv", index=False)
first_stage.to_csv(OUTPUT / "instrument_diagnostics.csv", index=False)
layer2_panel.to_csv(OUTPUT / "layer2_input.csv", index=False)
(OUTPUT / "fit_diagnostics.json").write_text(json.dumps(diagnostics, indent=2))

## Optional recovery check: evaluation only
Compare estimates with simulation truth only after completing estimation. Aggregate share fit is mechanical after inversion and is not evidence of parameter recovery.

In [ ]:
truth = json.loads((DATA / "true_parameters.json").read_text())["blp"]
mapping = {"price": "price_coefficient", "quality": "quality_mean"}
recovery = coefficients.copy()
recovery["true_value"] = [truth[mapping.get(k, k)] for k in recovery.parameter]
recovery["error"] = recovery.estimate - recovery.true_value
display(recovery)
recovery.to_csv(OUTPUT / "parameter_recovery.csv", index=False)